In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, recall_score, accuracy_score, precision_score, f1_score,classification_report,confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, auc


In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
os.getcwd()

'/home/euphrates/Desktop/dreps/DS/Radiomics/Boun-radiomics/notebooks/denemedosyasi/ozyegin-1/ozyegin'

In [5]:
os.listdir("CSV")

['rcc2.csv',
 'rcc1.csv',
 'test2.csv',
 'originalitself.csv',
 'flippedOriginal.csv',
 'phase_reg_key.csv',
 'test1.csv',
 'dilatedResized.csv',
 'erodedOriginal.csv',
 'dilatedOriginal.csv',
 'resizeditself.csv',
 'erodedResized.csv',
 'flippedResized.csv']

###  UCSF  --arterial  -- aml

In [6]:
reg = pd.read_csv('CSV/phase_reg_key.csv')


In [7]:
reg.columns

Index(['pid', 'noncon', 'arterial', 'delay', 'portven', 'mask',
       'ct_manufacturer', 'patient_age_years', 'patient_sex', 'pathology',
       'tumor_type', 'pathology_grade', 'tumor_laterality', 'tumor_size_cm',
       'tumor_biopsy_type'],
      dtype='object')

In [30]:
dd1= pd.read_csv('CSV/dilatedOriginal.csv')

In [31]:
dd1.shape

(8, 1180)

In [32]:
dd1['augmentation'] = 'dilated'
dd1['source_subset'] = 'Original'


In [33]:
pd.set_option("display.max_rows",1500)

In [34]:
dd1.iloc[1,:].to_dict().items()

dict_items([('PID', '45Nh52EJwK'), ('label', 'Unknown'), ('tumor_type', 2.0), ('pathology', 'angiomyolipoma'), ('pathology_grade', nan), ('Patient_Age', '058Y'), ('Patient_Sex', 'F'), ('Manufacturer', 'GE MEDICAL SYSTEMS LightSpeed VCT'), ('phase', 'arterial'), ('ct_spacing_mm', '[0.8183590173721313, 0.8183590173721313, 2.5]'), ('mask_spacing_mm', '[0.8183590173721313, 0.8183590173721313, 2.5]'), ('ct_shape', '[512, 512, 104]'), ('mask_voxel_count', 51331), ('original_shape_Elongation', 0.8780464769296084), ('original_shape_Flatness', 0.6951523973900751), ('original_shape_LeastAxisLength', 40.83170403922956), ('original_shape_MajorAxisLength', 58.73777346166213), ('original_shape_Maximum2DDiameterColumn', 66.40030120413611), ('original_shape_Maximum2DDiameterRow', 58.83026432033091), ('original_shape_Maximum2DDiameterSlice', 65.85590330410783), ('original_shape_Maximum3DDiameter', 73.62744053679987), ('original_shape_MeshVolume', 85475.875), ('original_shape_MinorAxisLength', 51.574495

In [35]:
dd2= pd.read_csv('CSV/dilatedResized.csv')

In [36]:
dd2.shape

(3, 1180)

In [37]:
dd2['augmentation'] = 'dilated'
dd2['source_subset'] = 'Resized'

In [38]:
df1= pd.read_csv('CSV/flippedOriginal.csv')

In [39]:
df1.shape

(8, 1180)

In [40]:
df1['augmentation'] = 'flipped'
df1['source_subset'] = 'Original'

In [41]:
df2 = pd.read_csv('CSV/flippedResized.csv')

In [42]:
df2.shape

(3, 1180)

In [43]:
df2['augmentation'] = 'flipped'
df2['source_subset'] = 'Resized'

In [44]:
de1 = pd.read_csv('CSV/erodedOriginal.csv')

In [45]:
de1.shape

(8, 1180)

In [46]:
de1['augmentation'] = 'eroded'
de1['source_subset'] = 'Original'

In [47]:
de2 = pd.read_csv('CSV/erodedResized.csv')

In [48]:
de2.shape

(3, 1180)

In [49]:
de2['augmentation'] = 'eroded'
de2['source_subset'] = 'Resized'

In [50]:
do = pd.read_csv('CSV/originalitself.csv')

In [51]:
do.shape

(8, 1180)

In [52]:
do['augmentation'] = 'NOT'
do['source_subset'] = 'Original'

In [53]:
dr = pd.read_csv('CSV/resizeditself.csv')

In [54]:
dr.shape

(3, 1180)

In [55]:
dr['augmentation'] = 'NOT'
dr['source_subset'] = 'Resized'

In [56]:
drcc1 = pd.read_csv('CSV/rcc1.csv')
drcc1.shape

(23, 1180)

In [57]:
drcc1['augmentation'] = 'NOT'
drcc1['source_subset'] = 'Original-RCC'

In [58]:
drcc2 = pd.read_csv('CSV/rcc2.csv')

In [59]:
drcc2.shape

(35, 1180)

In [60]:
drcc2['augmentation'] = 'NOT'
drcc2['source_subset'] = 'Original-RCC'

In [61]:
dfA= pd.concat([dd1,dd2,df1,df2,de1,de2,do,dr], axis = 0, ignore_index= 1)

In [62]:
dfA.shape

(44, 1182)

In [63]:
dfA['target'] = 1

In [64]:
dfR = pd.concat([drcc1,drcc2], axis = 0, ignore_index= 1)

In [65]:
dfR.shape

(58, 1182)

In [66]:
dfR['target'] = 0

In [67]:
dfA.columns

Index(['PID', 'label', 'tumor_type', 'pathology', 'pathology_grade',
       'Patient_Age', 'Patient_Sex', 'Manufacturer', 'phase', 'ct_spacing_mm',
       ...
       'diagnostics_Mask-interpolated_VoxelNum',
       'diagnostics_Mask-interpolated_VolumeNum',
       'diagnostics_Mask-interpolated_CenterOfMassIndex',
       'diagnostics_Mask-interpolated_CenterOfMass',
       'diagnostics_Mask-interpolated_Mean',
       'diagnostics_Mask-interpolated_Minimum',
       'diagnostics_Mask-interpolated_Maximum', 'augmentation',
       'source_subset', 'target'],
      dtype='object', length=1183)

In [68]:
dfA= pd.concat([do,dr], axis = 0, ignore_index= 1)

In [69]:
dfA['target'] = 1

In [70]:
dfA.shape

(11, 1183)

In [71]:
dfR = pd.concat([drcc1,drcc2], axis = 0, ignore_index= 1)

In [72]:
dfR.shape

(58, 1182)

In [73]:
dfR['target'] = 0

In [74]:
dfR.shape

(58, 1183)

In [75]:
dfrcc = dfR.sample(58)

In [76]:
dfU = pd.concat([dfA,dfrcc], axis = 0, ignore_index= 1)

In [77]:
dfU.drop('pathology_grade', axis =1, inplace= True)

In [78]:
y = dfU['target']

In [79]:
X = dfU.drop(columns=['augmentation','source_subset','target'],axis=1).select_dtypes(exclude = "object")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import SVC  # Swapped to Support Vector Machine Classifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import average_precision_score, make_scorer, confusion_matrix, accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# 1. Generate a mock small, imbalanced dataset (e.g., 168 samples, 20 features)
n_feat = 11
feature_names = [f"feature_{i}" for i in range(n_feat)]

X, y = make_classification(
    n_samples=168,
    n_features=n_feat,
    n_informative=8,
    n_classes=2,
    weights=[58/69, 11/69],
    flip_y=0,
    random_state=42
)

# 2. Define the Pipeline (SMOTE -> Feature Selection -> SVM Classifier)
# Crucial: probability=True is required to calculate predict_proba() for PR-AUC
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('selector', SelectKBest(score_func=f_classif)),
    ('classifier', SVC(
        random_state=42,
        class_weight='balanced',  # Automatically handles the 58:11 class imbalance
        probability=True          # Enables probability outputs for PR-AUC calculation
    ))
])

# 3. Define Hyperparameter Space for the Inner Loop (SVM Parameters)
param_grid = {
    'smote__k_neighbors': [3,4,5,6],
    'selector__k': [7, 10, 11, 13, 17, 19],
    'classifier__kernel': ['linear', 'rbf'],                    # Evaluate linear vs non-linear spaces
    'classifier__C': [0.1, 1, 10, 100],                         # Regularization strength
    'classifier__gamma': ['scale', 'auto', 0.01, 0.1]           # Kernel coefficient
}

# 4. Set up Inner and Outer Stratified K-Fold loops
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pr_auc_scorer = make_scorer(average_precision_score, response_method='predict_proba')

# 5. The Inner Loop Setup (Hyperparameter Tuning)
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=inner_cv,
    scoring=pr_auc_scorer,
    n_jobs=-1,
    error_score=0.0
)

# 6. The Outer Loop Execution (Model Evaluation)
outer_scores = []
outer_accuracy_scores = []
outer_f1_scores = []

# Storage units to capture data partitions to extract the best fold later
fold_data_registry = {}
best_fold_idx = -1
highest_pr_auc = -1.0

print("--- Starting Nested Cross-Validation (SVM) ---")
for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
    # Isolate the Outer Test Fold from the Outer Training Dataset
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Run the inner loop tuning on the outer training fold data only
    grid_search.fit(X_train, y_train)

    # Extract the best model from this fold's tuning configuration
    best_model = grid_search.best_estimator_

    # Predict probabilities and hard labels
    y_proba = best_model.predict_proba(X_test)[:, 1]
    y_pred = best_model.predict(X_test)

    # Evaluate performance on the real, imbalanced test fold
    fold_score = average_precision_score(y_test, y_proba)
    outer_scores.append(fold_score)

    fold_accuracy = accuracy_score(y_test, y_pred)
    outer_accuracy_scores.append(fold_accuracy)

    fold_f1 = f1_score(y_test, y_pred)
    outer_f1_scores.append(fold_f1)

    fold_cm = confusion_matrix(y_test, y_pred)

    # Cache the split data and fitted model for extraction
    fold_data_registry[fold] = {
        'X_train': X_train, 'y_train': y_train,
        'X_test': X_test, 'y_test': y_test,
        'fitted_pipeline': best_model
    }

    # Identify if this fold is currently outperforming previous ones
    if fold_score > highest_pr_auc:
        highest_pr_auc = fold_score
        best_fold_idx = fold

    print(f"Outer Fold {fold + 1} - Best Params: {grid_search.best_params_}")
    print(f"PR-AUC: {fold_score:.4f} | Accuracy: {fold_accuracy:.4f} | F1-Score: {fold_f1:.4f}")
    print(f"Confusion Matrix:\n{fold_cm}\n")

# 7. Aggregate and Final Verification
mean_pr_auc = np.mean(outer_scores)
std_pr_auc = np.std(outer_scores)

mean_accuracy = np.mean(outer_accuracy_scores)
std_accuracy = np.std(outer_accuracy_scores)

mean_f1 = np.mean(outer_f1_scores)
std_f1 = np.std(outer_f1_scores)

print("--- Final Performance Deliverable ---")
print(f"Nested CV Mean PR-AUC:   {mean_pr_auc:.4f} +/- {std_pr_auc:.4f}")
print(f"Nested CV Mean Accuracy: {mean_accuracy:.4f} +/- {std_accuracy:.4f}")
print(f"Nested CV Mean F1-Score: {mean_f1:.4f} +/- {std_f1:.4f}\n")